In [1]:
from pathlib import Path
import pandas as pd
from PyDI.io import load_parquet
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 25)
pd.set_option('display.max_colwidth', 100)

ROOT = Path.cwd()

DATA_DIR = ROOT / "parquet"
OUTPUT_DIR = ROOT / "output"
MLDS_DIR = ROOT / "ml-datasets"
BLOCK_EVAL_DIR = OUTPUT_DIR / "blocking_evaluation"
CORR_DIR = OUTPUT_DIR / "correspondences"

PIPELINE_DIR = OUTPUT_DIR / "data_fusion"
PIPELINE_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
amazon_sample = load_parquet(
    DATA_DIR / "amazon_sample.parquet",
    name="amazon_sample"
)

goodreads_sample = load_parquet(
    DATA_DIR / "goodreads_sample.parquet",
    name="goodreads_sample"
)

metabooks_sample = load_parquet(
  DATA_DIR / "metabooks_sample.parquet",
  name="metabooks_sample"
)

In [3]:
from PyDI.io import load_csv

train_m2a = load_csv(
    MLDS_DIR / "train_MA.csv",
    name="train_metabooks_amazon",
    header=0,
    names=['id1', 'id2', 'label'],
    add_index=False
)

test_m2a = load_csv(
    MLDS_DIR / "test_MA.csv",
    name="test_metabooks_amazon",
    header=0,
    names=['id1', 'id2', 'label'],
    add_index=False
)

train_m2g = load_csv(
    MLDS_DIR / "train_MG.csv",
    name="train_metabooks_goodreads",
    header=0,
    names=['id1', 'id2', 'label'],
    add_index=False
)

test_m2g = load_csv(
    MLDS_DIR / "test_MG.csv",
    name="test_metabooks_goodreads",
    header=0,
    names=['id1', 'id2', 'label'],
    add_index=False
)

In [4]:
from PyDI.entitymatching import EmbeddingBlocker

embedding_blocker_m2a = EmbeddingBlocker(
    metabooks_sample, amazon_sample,
    text_cols=['title', 'author'],
    model="sentence-transformers/all-MiniLM-L6-v2",
    index_backend="sklearn",
    top_k=20,
    batch_size=200,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)


embedding_blocker_m2g = EmbeddingBlocker(
    metabooks_sample, goodreads_sample,
    text_cols=['title', 'author'],
    model="sentence-transformers/all-MiniLM-L6-v2",
    index_backend="sklearn",
    top_k=20,
    batch_size=200,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)

embedding_candidates_m2a = embedding_blocker_m2a.materialize()
embedding_candidates_m2g = embedding_blocker_m2g.materialize()

In [5]:
from PyDI.entitymatching import EntityMatchingEvaluator


# evaluate blocking: metabooks -> amazon
results_m2a = EntityMatchingEvaluator.evaluate_blocking(
    candidate_pairs=embedding_candidates_m2a,
    blocker=embedding_blocker_m2a,
    test_pairs=train_m2a,
    out_dir=BLOCK_EVAL_DIR
)


# evaluate blocking: metabooks -> amazon
results_m2g = EntityMatchingEvaluator.evaluate_blocking(
    candidate_pairs=embedding_candidates_m2g,
    blocker=embedding_blocker_m2g,
    test_pairs=train_m2g,
    out_dir=BLOCK_EVAL_DIR
)

display(results_m2a)
display(results_m2g)

{'pair_completeness': 0.9945652173913043,
 'pair_quality': 0.00026165321940694794,
 'reduction_ratio': 0.9994290620408163,
 'total_candidates': 699399,
 'total_possible_pairs': 1225000000,
 'true_positives_found': 183,
 'total_true_pairs': 184,
 'evaluation_timestamp': '2025-11-18T14:00:55.865263',
 'output_files': ['/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/blocking_evaluation/blocking_evaluation_summary.json',
  '/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/blocking_evaluation/blocking_detailed_results.csv']}

{'pair_completeness': 0.9875,
 'pair_quality': 0.0002265703981472851,
 'reduction_ratio': 0.9994307306122449,
 'total_candidates': 697355,
 'total_possible_pairs': 1225000000,
 'true_positives_found': 158,
 'total_true_pairs': 160,
 'evaluation_timestamp': '2025-11-18T14:01:27.379875',
 'output_files': ['/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/blocking_evaluation/blocking_evaluation_summary.json',
  '/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/blocking_evaluation/blocking_detailed_results.csv']}

In [187]:
from PyDI.entitymatching import StringComparator, NumericComparator

comparators = [
    StringComparator(column='title',similarity_function='cosine'),
    StringComparator(column='title',tokenization="char",similarity_function='jaccard'),
    StringComparator(column='title',similarity_function='jaro_winkler'),

    StringComparator(column='author',similarity_function='cosine', ),
    StringComparator(column='author',similarity_function='jaccard', ),


    StringComparator(column='publisher',similarity_function='jaro_winkler', preprocess=str.lower),
    StringComparator(column='publisher',similarity_function='cosine', preprocess=str.lower),
    

    NumericComparator(column='publish_year',max_difference=1.0),


    NumericComparator(column="page_count", max_difference=10),
]

In [188]:
from PyDI.entitymatching import FeatureExtractor

feature_extractor = FeatureExtractor(comparators)

train_features_m2a = feature_extractor.create_features(
    metabooks_sample, amazon_sample, train_m2a[['id1', 'id2']], labels=train_m2a['label'], id_column='id'
)

train_features_m2g = feature_extractor.create_features(
    metabooks_sample, goodreads_sample, train_m2g[['id1', 'id2']], labels=train_m2g['label'], id_column='id'
)

feature_columns_m2a = [col for col in train_features_m2a.columns if col not in ['id1', 'id2', 'label']]
X_train_m2a = train_features_m2a[feature_columns_m2a]
y_train_m2a = train_features_m2a['label']

feature_columns_m2g = [col for col in train_features_m2g.columns if col not in ['id1', 'id2', 'label']]
X_train_m2g = train_features_m2g[feature_columns_m2g]
y_train_m2g = train_features_m2g['label']

training_datasets = [(X_train_m2a, y_train_m2a),(X_train_m2g, y_train_m2g)]

In [189]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer, f1_score
import pandas as pd
import numpy as np
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# classifiers
classifiers = {
    'RandomForestClassifier': RandomForestClassifier(random_state=42),
    'GradientBoostingClassifier': GradientBoostingClassifier(random_state=42),
    'SVC': SVC(probability=True, random_state=42),
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42)
}

# parameter grids
param_grids = {
    'RandomForestClassifier': {
        'n_estimators': [50, 100, 200, 500],
        'max_depth': [None, 10, 20],
        'class_weight': ['balanced', None],
        'min_samples_split': [2, 5]
    },
    'GradientBoostingClassifier': {
        'n_estimators': [100, 200],
        'learning_rate': [0.05, 0.1],
        'max_depth': [3, 5]
    },
    'SVC': {
        'C': [0.1, 1, 10],
        'kernel': ['linear', 'rbf'],
        'gamma': ['scale', 'auto']
    },
    'LogisticRegression': {
        'C': [0.01, 0.1, 1, 10],
        'penalty': ['l2'],
        'solver': ['lbfgs', 'liblinear']
    }
}

scorer = make_scorer(f1_score)
cv_folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

training_datasets = [
    (X_train_m2a, y_train_m2a),
    (X_train_m2g, y_train_m2g),
]

best_models = []  # one best model per dataset

for (X_train, y_train) in training_datasets:
    grid_search_results = {}
    best_overall_score = -1
    best_overall_model = None
    best_model_name = None

    for name, model in classifiers.items():
        print(f"Running GridSearchCV for {name}...")
        
        grid = GridSearchCV(
            estimator=model,
            param_grid=param_grids[name],
            scoring=scorer,
            cv=cv_folds,
            n_jobs=-1,
            verbose=0
        )
        
        grid.fit(X_train, y_train)
        print(
            f"{name}: best F1 = {grid.best_score_:.4f} "
            f"with params {grid.best_params_}"
        )
        
        grid_search_results[name] = {
            'grid_search': grid,
            'best_score': grid.best_score_,
            'best_params': grid.best_params_,
            'best_estimator': grid.best_estimator_
        }

        if grid.best_score_ > best_overall_score:
            best_overall_score = grid.best_score_   
            best_overall_model = grid.best_estimator_
            best_model_name = name

    print(f"Best model for this dataset: {best_model_name} with F1={best_overall_score:.4f}")
    best_models.append(best_overall_model)

Running GridSearchCV for RandomForestClassifier...


RandomForestClassifier: best F1 = 0.8906 with params {'class_weight': None, 'max_depth': 10, 'min_samples_split': 5, 'n_estimators': 50}
Running GridSearchCV for GradientBoostingClassifier...
GradientBoostingClassifier: best F1 = 0.8855 with params {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 200}
Running GridSearchCV for SVC...
SVC: best F1 = 0.8886 with params {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}
Running GridSearchCV for LogisticRegression...
LogisticRegression: best F1 = 0.8866 with params {'C': 10, 'penalty': 'l2', 'solver': 'lbfgs'}
Best model for this dataset: RandomForestClassifier with F1=0.8906
Running GridSearchCV for RandomForestClassifier...
RandomForestClassifier: best F1 = 0.9285 with params {'class_weight': 'balanced', 'max_depth': None, 'min_samples_split': 5, 'n_estimators': 100}
Running GridSearchCV for GradientBoostingClassifier...
GradientBoostingClassifier: best F1 = 0.9224 with params {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 200}
R

In [191]:
best_models

[RandomForestClassifier(max_depth=10, min_samples_split=5, n_estimators=50,
                        random_state=42),
 SVC(C=10, probability=True, random_state=42)]

In [192]:
import numpy as np
misclassified_rows = []  # list of DataFrames, one per dataset

for (X_train, y_train), model in zip(training_datasets, best_models):
    # Predict on the same training data
    y_pred = model.predict(X_train)

    # Boolean mask of incorrect predictions
    mis_mask = (y_pred != y_train)

    # Build a DataFrame of misclassified rows
    X_df = pd.DataFrame(X_train)  # in case X_train is a numpy array
    errors_df = X_df[mis_mask].copy()
    errors_df["y_true"] = np.array(y_train)[mis_mask]
    errors_df["y_pred"] = y_pred[mis_mask]

    misclassified_rows.append(errors_df)

In [198]:
misclassified_rows[0]

,"StringComparator(title, cosine, tokenization=word, list_strategy=None)","StringComparator(title, jaccard, tokenization=char, list_strategy=None)","StringComparator(title, jaro_winkler, tokenization=char, list_strategy=None)","StringComparator(author, cosine, tokenization=word, list_strategy=None)","StringComparator(author, jaccard, tokenization=word, list_strategy=None)","StringComparator(publisher, jaro_winkler, tokenization=char, list_strategy=None)","StringComparator(publisher, cosine, tokenization=word, list_strategy=None)","NumericComparator(publish_year, absolute_difference, list_strategy=None)","NumericComparator(page_count, absolute_difference, list_strategy=None)",y_true,y_pred
653,0.408248,0.422222,0.822391,1.0,1.0,0.941667,0.666667,0.0,0.0,1,0
683,0.492366,0.440000,0.831071,1.0,1.0,1.000000,1.000000,0.0,0.0,1,0
694,0.503953,0.603448,0.874831,1.0,1.0,1.000000,1.000000,0.0,0.0,1,0
708,0.447214,0.484848,0.857845,1.0,1.0,0.830769,0.353553,0.0,0.0,1,0
726,0.577350,0.514286,0.866869,1.0,1.0,0.850962,0.500000,0.0,0.0,1,0
750,0.507093,0.552632,0.888811,1.0,1.0,1.000000,1.000000,0.0,0.0,1,0
779,0.600000,0.702128,0.893843,1.0,1.0,0.631011,0.000000,0.0,0.0,1,0


In [199]:
misclassified_rows[1]

,"StringComparator(title, cosine, tokenization=word, list_strategy=None)","StringComparator(title, jaccard, tokenization=char, list_strategy=None)","StringComparator(title, jaro_winkler, tokenization=char, list_strategy=None)","StringComparator(author, cosine, tokenization=word, list_strategy=None)","StringComparator(author, jaccard, tokenization=word, list_strategy=None)","StringComparator(publisher, jaro_winkler, tokenization=char, list_strategy=None)","StringComparator(publisher, cosine, tokenization=word, list_strategy=None)","NumericComparator(publish_year, absolute_difference, list_strategy=None)","NumericComparator(page_count, absolute_difference, list_strategy=None)",y_true,y_pred
350,0.745356,0.560976,0.912195,0.288675,0.142857,1.000000,1.000000,0.0,1.0,0,1
429,0.714286,0.571429,0.881005,0.534522,0.285714,1.000000,1.000000,0.0,1.0,0,1
646,0.447214,0.333333,0.866667,0.707107,0.500000,0.644024,0.267261,0.0,0.0,1,0
679,0.000000,0.195122,0.592130,0.707107,0.500000,0.669697,0.316228,0.0,0.0,1,0
680,0.534522,0.277778,0.855556,0.774597,0.600000,0.514550,0.000000,0.0,0.0,1,0
693,0.333333,0.700000,0.776144,0.577350,0.333333,0.296825,0.000000,0.0,0.0,1,0
726,0.577350,0.500000,0.900000,0.816497,0.666667,1.000000,1.000000,0.0,0.0,1,0
735,0.577350,0.529412,0.905882,0.707107,0.500000,1.000000,1.000000,0.0,1.0,1,0
756,0.522233,0.333333,0.866667,1.000000,1.000000,1.000000,1.000000,0.0,0.0,1,0
765,0.707107,0.600000,0.920000,1.000000,1.000000,0.876190,0.577350,0.0,0.0,1,0


In [229]:
from PyDI.entitymatching import MLBasedMatcher


ml_matcher = MLBasedMatcher(feature_extractor)

correspondences_m2a = ml_matcher.match(
    metabooks_sample, amazon_sample,
    candidates=embedding_blocker_m2a,
    id_column='id',
    trained_classifier=best_models[0],
)

correspondences_m2g = ml_matcher.match(
    metabooks_sample, goodreads_sample,
    candidates=embedding_blocker_m2g,
    id_column='id',
    trained_classifier=best_models[1]
)

In [230]:
debug_output_dir = OUTPUT_DIR / "debug_results_entity_matching"
debug_output_dir.mkdir(parents=True, exist_ok=True)

# evaluate matching metabooks -> amazon
eval_results_m2a = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_m2a,
    test_pairs=test_m2a,
    out_dir=debug_output_dir
)

display(eval_results_m2a)

{'precision': 0.9069767441860465,
 'recall': 0.8478260869565217,
 'f1': 0.8764044943820224,
 'accuracy': 0.945,
 'true_positives': 39,
 'false_positives': 4,
 'false_negatives': 7,
 'true_negatives': 150,
 'threshold_used': 0.0,
 'total_correspondences': 36687,
 'filtered_correspondences': 36687,
 'evaluation_timestamp': '2025-11-18T17:58:44.952792',
 'output_files': ['/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/debug_results_entity_matching/matching_evaluation_summary.json',
  '/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/debug_results_entity_matching/matching_detailed_results.csv']}

In [231]:
# evaluate matching metabooks -> goodreads
eval_results_m2g = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_m2g,
    test_pairs=test_m2g,
    out_dir=debug_output_dir
)

display(eval_results_m2g)

{'precision': 0.972972972972973,
 'recall': 0.9,
 'f1': 0.935064935064935,
 'accuracy': 0.975,
 'true_positives': 36,
 'false_positives': 1,
 'false_negatives': 4,
 'true_negatives': 159,
 'threshold_used': 0.0,
 'total_correspondences': 10017,
 'filtered_correspondences': 10017,
 'evaluation_timestamp': '2025-11-18T17:58:59.364344',
 'output_files': ['/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/debug_results_entity_matching/matching_evaluation_summary.json',
  '/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/debug_results_entity_matching/matching_detailed_results.csv']}

In [232]:
cluster_analysis_dir = OUTPUT_DIR / "cluster_analysis"
cluster_analysis_dir.mkdir(parents=True, exist_ok=True)
cluster_distribution_m2a = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2a,
    out_dir=cluster_analysis_dir
)
cluster_distribution_m2g = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2g,
    out_dir=cluster_analysis_dir
)
print(f"\n📊 Cluster Size Distribution Results (Metabooks -> Amazon):")
display(cluster_distribution_m2a)
print(f"\n📊 Cluster Size Distribution Results (Metabooks -> Goodreads):")
display(cluster_distribution_m2g)


📊 Cluster Size Distribution Results (Metabooks -> Amazon):


,cluster_size,frequency,percentage
0,2,18231,84.755927
1,3,919,4.272431
2,4,1421,6.606230
3,5,231,1.073919
4,6,243,1.129707
...,...,...,...
43,85,1,0.004649
44,95,1,0.004649
45,99,2,0.009298
46,114,1,0.004649



📊 Cluster Size Distribution Results (Metabooks -> Goodreads):


,cluster_size,frequency,percentage
0,2,5743,81.960896
1,3,823,11.745397
2,4,212,3.025546
3,5,85,1.213073
4,6,50,0.713572
...,...,...,...
21,24,2,0.028543
22,27,1,0.014271
23,39,1,0.014271
24,40,1,0.014271


In [233]:
from PyDI.entitymatching import MaximumBipartiteMatching, StableMatching

# We are using Maxmimum Bipartite Matching to refine results to 1:1 matches
clusterer = MaximumBipartiteMatching()
mbm_correspondences_m2a = clusterer.cluster(correspondences_m2a)
mbm_correspondences_m2g = clusterer.cluster(correspondences_m2g)
all_correspondences = pd.concat([mbm_correspondences_m2a, mbm_correspondences_m2g], ignore_index=True)
len(all_correspondences)

33782

In [234]:
from PyDI.fusion import DataFusionStrategy, DataFusionEngine, longest_string, union, prefer_higher_trust
import pandas as pd

metabooks_sample.attrs["trust_score"] = 3
goodreads_sample.attrs["trust_score"] = 2
amazon_sample.attrs["trust_score"] = 1
strategy = DataFusionStrategy('books_fusion_strategy')

strategy.add_attribute_fuser('title', longest_string)
strategy.add_attribute_fuser('author', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('publish_year', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('publisher', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('language', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('price', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('page_count', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('numratings', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('genres',union)

engine = DataFusionEngine(strategy, debug=True, debug_format='json',
                          debug_file= PIPELINE_DIR / "debug_fusion_ml_embedding_blocker.jsonl")

fused_ml_emblocker = engine.run(
    datasets=[amazon_sample, metabooks_sample, goodreads_sample],
    correspondences=all_correspondences,
    id_column="id",
    include_singletons=False
)
fused_ml_emblocker.to_parquet(PIPELINE_DIR / "fused_ml_emblocker.parquet")
print(f'Fused rows: {len(fused_ml_emblocker):,}')

Fused rows: 30,183


In [235]:
from PyDI.fusion import tokenized_match, boolean_match,numeric_tolerance_match,set_equality_match

import numpy as np
import re, ast, numpy as np, pandas as pd


def categories_set_equal(a, b) -> bool:
    """Return True if a and b contain the same unique categories (order/type agnostic)."""
    def to_set(x):
        def items(v):
            # missing
            if v is None or (isinstance(v, float) and np.isnan(v)): return []
            # numpy array → recurse over elements
            if isinstance(v, np.ndarray): 
                out=[]; [out.extend(items(e)) for e in v.flatten()]; return out
            # python containers → recurse over elements
            if isinstance(v, (list, tuple, set)):
                out=[]; [out.extend(items(e)) for e in v]; return out
            # scalar/string: try parse stringified list; else split by delimiters
            s = str(v).strip()
            if s == "" or s.lower() in {"nan","none"}: return []
            try:
                parsed = ast.literal_eval(s)
                if isinstance(parsed, (list, tuple, set)): return items(parsed)
            except Exception:
                pass
            return [p.strip() for p in re.split(r"[|,;/]", s) if p.strip()]
        return {it.lower() for it in items(x)}
    return to_set(a) == to_set(b)

strategy.add_evaluation_function("title", tokenized_match)
strategy.add_evaluation_function("author", tokenized_match)
strategy.add_evaluation_function("publisher", tokenized_match)
strategy.add_evaluation_function("publish_year", numeric_tolerance_match)
strategy.add_evaluation_function("numratings", numeric_tolerance_match)
strategy.add_evaluation_function("price", numeric_tolerance_match)
strategy.add_evaluation_function("page_count", numeric_tolerance_match)
strategy.add_evaluation_function("language", tokenized_match)
strategy.add_evaluation_function("genres", categories_set_equal)

fused_dataset = pd.read_parquet(PIPELINE_DIR / "fused_ml_emblocker.parquet")
fused_dataset["publish_year"] = fused_dataset["publish_year"].astype("Int16")
fused_dataset["page_count"] = fused_dataset["page_count"].astype("Int32")
golden_fused_dataset= pd.read_parquet(MLDS_DIR / "golden_fused_books.parquet")

In [236]:
from PyDI.fusion import DataFusionEvaluator
fused_dataset.drop_duplicates(subset='isbn_clean', keep='first',inplace=True)
# Create evaluator with our fusion strategy
evaluator = DataFusionEvaluator(strategy, debug=True, debug_file=OUTPUT_DIR / "data_fusion" / "debug_fusion_eval.jsonl", debug_format="json")

# Evaluate the fused results against the gold standard
print("Evaluating fusion results against gold standard...")
evaluation_results = evaluator.evaluate(
    fused_df=fused_dataset,
    fused_id_column='isbn_clean',
    gold_df=golden_fused_dataset,
    gold_id_column='isbn_clean',
)

# Display evaluation metrics
print("\nFusion Evaluation Results:")
print("=" * 40)
for metric, value in evaluation_results.items():
    if isinstance(value, float):
        print(f"  {metric}: {value:.3f}")
    else:
        print(f"  {metric}: {value}")
        
print(f"\nOverall Accuracy: {evaluation_results.get('overall_accuracy', 0):.1%}")

Evaluating fusion results against gold standard...

Fusion Evaluation Results:
  overall_accuracy: 0.716
  macro_accuracy: 0.716
  num_evaluated_records: 38828
  num_evaluated_attributes: 9
  total_evaluations: 342014
  total_correct: 244973
  numratings_accuracy: 0.727
  numratings_count: 38828
  genres_accuracy: 0.684
  genres_count: 37448
  language_accuracy: 0.749
  language_count: 38695
  title_accuracy: 0.723
  title_count: 38828
  publish_year_accuracy: 0.710
  publish_year_count: 38708
  page_count_accuracy: 0.715
  page_count_count: 35411
  price_accuracy: 0.714
  price_count: 36440
  publisher_accuracy: 0.729
  publisher_count: 38828
  author_accuracy: 0.695
  author_count: 38828

Overall Accuracy: 71.6%


In [24]:
metabooks_sample[metabooks_sample["isbn_clean"] == "034525483X"]

,id,title,author,rating,numratings,language,genres,bookformat,edition,page_count,publisher,publish_year,price,isbn_clean
6279,metabooks_102449,the princess bride,william goldman,4.7,15444,English,"Science Fiction, Fantasy",None,None,<NA>,Ballantine Books,<NA>,69.620003,034525483X


In [29]:
golden = golden_fused_dataset[["isbn_clean", "title"]].rename(
    columns={"title": "title_golden"}
)
fused = fused_dataset[["isbn_clean", "title"]].rename(
    columns={"title": "title_fused"}
)

merged = golden.merge(fused, on="isbn_clean", how="inner")
diff = merged[merged["title_golden"] != merged["title_fused"]]

diff[["isbn_clean", "title_golden","title_fused"]].head()


,isbn_clean,title_golden,title_fused
4,0002160595,c s lewis a biography,jack a life of c s lewis
125,0020259913,the empire of the atom collier nucleus science fiction classic,the course of empire
128,0020292651,be expert with map and compass the complete orienteering handbook,be expert with map and compass the complete orienteering handbook emblem editions
131,002029865X,the steps of the sun,dark sun the making of the hydrogen bomb
159,0023895306,nicomachean ethics aristotle,nicomachean ethics


In [37]:
datasets = [
    ("amazon", amazon_sample),
    ("goodreads", goodreads_sample),
    ("metabooks", metabooks_sample),
    ("fused", fused_dataset),
    ("golden", golden_fused_dataset),
]

def find_title(term: str, column: str = "title"):
    for name, df in datasets:
        hits = df[df[column].str.contains(term, case=False, na=False)]
        if not hits.empty:
            print(f"=== {name} ({len(hits)} matches) ===")
            display(hits.head())  # or print(hits.head())

# usage
find_title("0684142708","isbn_clean")


=== amazon (1 matches) ===


,id,title,author,rating,numratings,language,genres,bookformat,edition,page_count,publisher,publish_year,price,isbn_clean
17532,amazon_155957,be expert with map and compass the complete orienteering handbook emblem editions,bjorn kjellstrom,None,None,None,None,None,None,None,Macmillan General Reference,1976,None,0684142708


=== metabooks (1 matches) ===


,id,title,author,rating,numratings,language,genres,bookformat,edition,page_count,publisher,publish_year,price,isbn_clean
9065,metabooks_147691,be expert with map compass the complete orienteering handbook emblem editions,bjorn kjellstrom,4.1,26,English,,None,None,214,Charles Scribner's Sons,<NA>,14.39,0684142708


=== golden (1 matches) ===


,isbn_clean,title,author,publisher,publish_year,language,numratings,genres,page_count,price
23653,0684142708,be expert with map and compass the complete orienteering handbook emblem editions,bjorn kjellstrom,Charles Scribner's Sons,1976,English,26,None,214,14.39
